###### Lab 10: ANN and DNN Lab with Tensorflow

#### Step-by-Step Instructions:


In [ ]:
# Hello everyone! in this lab, we'll walk through the process of building and training an ANN and DNN using TensorFlow.
# We'll use pandas for data manipulation, matplotlib and seaborn for plotting, and TensorFlow and Keras for building and training the ANN.


#visit the links below to learn more about these libraries and install them before starting the lab
#https://www.tensorflow.org/
#https://keras.io/


import pandas as pd  # Importing pandas for data manipulation
import numpy as np  # Importing numpy for numerical operations
import tensorflow as tf  # Importing TensorFlow for building and training the ANN
from tensorflow.keras import layers  # Importing layers from TensorFlow's Keras API
from sklearn.model_selection import train_test_split  # Importing train_test_split from sklearn for splitting the data
from sklearn.preprocessing import StandardScaler  # Importing StandardScaler from sklearn for feature scaling
from sklearn.metrics import confusion_matrix, accuracy_score  # Importing evaluation metrics from sklearn
import matplotlib.pyplot as plt  # Importing matplotlib for plotting
import seaborn as sns  # Importing seaborn for advanced visualizations
import warnings  # Importing warnings to manage warning messages

# We'll ignore any warnings to keep our output clean.
warnings.filterwarnings('ignore')  # This will suppress any warning messages

# Let's start by loading our dataset into a pandas DataFrame.
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'  # URL for the dataset
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model Year', 'Origin']  # Defining column names
raw_dataset = pd.read_csv(url, names=column_names, na_values='?', comment='\t', sep=' ', skipinitialspace=True)  # Loading the dataset into a DataFrame

# Displaying the first few rows of the dataset to get an idea of what it looks like.
print(raw_dataset.head())  # Displaying the first few rows of the DataFrame

# Creating a copy of the dataset for further manipulation.
dataset = raw_dataset.copy()  # Making a copy of the DataFrame

# Displaying the last few rows of the dataset.
print(dataset.tail())  # Displaying the last few rows of the DataFrame

# Checking for missing values in the dataset.
print(dataset.isna().sum())  # Counting the number of missing values in each column

# Dropping rows with missing values.
dataset = dataset.dropna()  # Removing rows with missing values

# Confirming that there are no missing values left.
print(dataset.isna().sum())  # Counting the number of missing values in each column

# Converting the 'Origin' column to categorical labels.
dataset['Origin'] = dataset['Origin'].map({1: 'USA', 2: 'Europe', 3: 'Japan'})  # Mapping numeric values to country names

# One-hot encoding the categorical 'Origin' column.
dataset = pd.get_dummies(dataset, columns=['Origin'])  # Converting categorical column into dummy/indicator variables

# Splitting the dataset into training and testing sets.
train_dataset = dataset.sample(frac=0.8, random_state=0)  # Sampling 80% of the data for training
test_dataset = dataset.drop(train_dataset.index)  # Using the remaining 20% for testing

# Plotting pairwise relationships in the training dataset.
sns.pairplot(train_dataset[['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model Year']])  # Visualizing relationships

# Displaying summary statistics of the training dataset.
print(train_dataset.describe().transpose())  # Showing summary statistics

# Separating features and labels.
train_features = train_dataset.copy()  # Making a copy of the training dataset for features
test_features = test_dataset.copy()  # Making a copy of the testing dataset for features
train_labels = train_features.pop('MPG')  # Removing the label column from the training features
test_labels = test_features.pop('MPG')  # Removing the label column from the testing features

# Normalizing the features.
normalizer = tf.keras.layers.Normalization(axis=-1)  # Creating a normalization layer
normalizer.adapt(np.array(train_features))  # Adapting the normalization layer to the training features

# Printing the mean values used for normalization.
print(normalizer.mean.numpy())  # Displaying the mean values used for normalization

# Building a simple linear model for horsepower.
horsepower = np.array(train_features['Horsepower'])  # Extracting the horsepower feature
horsepower_normalizer = layers.Normalization(input_shape=[1,], axis=None)  # Creating a normalization layer for horsepower
horsepower_normalizer.adapt(horsepower)  # Adapting the normalization layer to the horsepower feature

# Creating a sequential model with a normalization layer and a dense layer.
horsepower_model = tf.keras.Sequential([
    horsepower_normalizer,  # Adding the normalization layer
    layers.Dense(units=1)  # Adding a dense layer with one unit
])

# Displaying the model summary.
horsepower_model.summary()  # Showing the model architecture

# Compiling the model with Adam optimizer and mean absolute error loss.
horsepower_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),  # Using Adam optimizer
    loss='mean_absolute_error'  # Using mean absolute error as the loss function
)

# Training the model.
history = horsepower_model.fit(
    train_features['Horsepower'],  # Training on horsepower feature
    train_labels,  # Using MPG as the label
    epochs=100,  # Training for 100 epochs
    verbose=0,  # Suppressing verbose output
    validation_split=0.2  # Using 20% of the data for validation
)

# Converting the training history to a DataFrame.
hist = pd.DataFrame(history.history)  # Creating a DataFrame from the training history
hist['epoch'] = history.epoch  # Adding an epoch column
print(hist.tail())  # Displaying the last few rows of the DataFrame

# Defining a function to plot the training and validation loss.
def plot_loss(history):
    plt.plot(history.history['loss'], label='loss')  # Plotting training loss
    plt.plot(history.history['val_loss'], label='val_loss')  # Plotting validation loss
    plt.ylim([0, 10])  # Setting y-axis limits
    plt.xlabel('Epoch')  # Labeling the x-axis
    plt.ylabel('Error [MPG]')  # Labeling the y-axis
    plt.legend()  # Adding a legend
    plt.grid(True)  # Adding a grid

# Plotting the training and validation loss.
plot_loss(history)  # Calling the function to plot loss

# Making predictions using the trained model.
x = tf.linspace(0.0, 250, 251)  # Creating a range of values for horsepower
y = horsepower_model.predict(x)  # Predicting MPG using the model

# Defining a function to plot horsepower vs. MPG.
def plot_horsepower(x, y):
    plt.scatter(train_features['Horsepower'], train_labels, label='Data')  # Plotting the training data
    plt.plot(x, y, color='k', label='Predictions')  # Plotting the model predictions
    plt.xlabel('Horsepower')  # Labeling the x-axis
    plt.ylabel('MPG')  # Labeling the y-axis
    plt.legend()  # Adding a legend

# Plotting horsepower vs. MPG.
plot_horsepower(x, y)  # Calling the function to plot horsepower vs. MPG

# Building and training a linear model using all features.
linear_model = tf.keras.Sequential([
    normalizer,  # Adding the normalization layer
    layers.Dense(units=1)  # Adding a dense layer with one unit
])

# Compiling the linear model with Adam optimizer and mean absolute error loss.
linear_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),  # Using Adam optimizer
    loss='mean_absolute_error'  # Using mean absolute error as the loss function
)

# Training the linear model.
history = linear_model.fit(
    train_features,  # Using all features for training
    train_labels,  # Using MPG as the label
    epochs=100,  # Training for 100 epochs
    verbose=0,  # Suppressing verbose output
    validation_split=0.2  # Using 20% of the data for validation
)

# Plotting the training and validation loss for the linear model.
plot_loss(history)  # Calling the function to plot aloss

# Evaluating the linear model on the test data.
test_results = {}
test_results['linear_model'] = linear_model.evaluate(test_features, test_labels, verbose=0)  # Evaluating the model

# Building and training a deep neural network (DNN) model with one feature.
dnn_horsepower_model = tf.keras.Sequential([
    horsepower_normalizer,  # Adding the normalization layer
    layers.Dense(64, activation='relu'),  # Adding a dense layer with 64 units and ReLU activation
    layers.Dense(64, activation='relu'),  # Adding another dense layer with 64
])

    MPG  Cylinders  Displacement  Horsepower  Weight  Acceleration  \
0  18.0          8         307.0       130.0  3504.0          12.0   
1  15.0          8         350.0       165.0  3693.0          11.5   
2  18.0          8         318.0       150.0  3436.0          11.0   
3  16.0          8         304.0       150.0  3433.0          12.0   
4  17.0          8         302.0       140.0  3449.0          10.5   

   Model Year  Origin  
0          70       1  
1          70       1  
2          70       1  
3          70       1  
4          70       1  
      MPG  Cylinders  Displacement  Horsepower  Weight  Acceleration  \
393  27.0          4         140.0        86.0  2790.0          15.6   
394  44.0          4          97.0        52.0  2130.0          24.6   
395  32.0          4         135.0        84.0  2295.0          11.6   
396  28.0          4         120.0        79.0  2625.0          18.6   
397  31.0          4         119.0        82.0  2720.0          19.4   

  

ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type int).

In [ ]:
# MIT License
#
# Copyright (c) 2017 François Chollet
#
# Permission is hereby granted, free of charge, to any person obtaining a
# copy of this software and associated documentation files (the "Software"),
# to deal in the Software without restriction, including without limitation
# the rights to use, copy, modify, merge, publish, distribute, sublicense,
# and/or sell copies of the Software, and to permit persons to whom the
# Software is furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in
# all copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL
# THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING
# FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER
# DEALINGS IN THE SOFTWARE.